# Exploratory Notebook : Data Engineering
## Introduction
We are going to build a RAG system for the French company Calogena. In this notebook, we will:
- examine the publicly available material regarding the company
- check if the extracted texts are clean and make corrections if necessary
- determine the chunk size for the RAG system
- select the appropriate embedding model
- test our assistant

In [136]:
from pathlib import Path
import urllib.request
from pypdf import PdfReader
import pandas as pd 
from collections import Counter 
import re 
import json

## 01. ASNR

To begin with, we will look for the documents needed for our RAG system.            

We require reliable documents containing verified information. We also need a substantial amount of content for our embedding process to work effectively. We will therefore turn to the ASN, the Nuclear Safety Authority, which has published sourced documents, each around ten pages long regarding Calogena.

Here is the first PDF, the 2026 ASNR expert assessment: It evaluates the CAL-30, the reactor Calogena intends to use, which features two emergency shutdown systems. The document focuses on these systems.       
[Avis d'expertise ASNR 2026](https://recherche-expertise.asnr.fr/sites/default/files/2026-06/ASNR-avis-expertise-2026-00015.pdf)

The second is the 2026 Board opinion: This represents the official position based on the expert assessment. It addresses the same emergency shutdown systems, deeming them satisfactory at this stage, while calling for a simple, robust alternative method.    
[Avis ASNR 2026](https://reglementation-controle.asnr.fr/content/download/209769/file/2026-AV-027.pdf)   

Both of these PDFs are relatively short, about ten pages each. We will look at the implications of this later.

So we are going to download them.       
[Python Documentation on urllib](https://docs.python.org/3/library/urllib.request.html)

In [75]:
raw_path = Path("data/raw")

# 2026 expert assessment
urllib.request.urlretrieve(
    "https://recherche-expertise.asnr.fr/sites/default/files/2026-06/ASNR-avis-expertise-2026-00015.pdf",
    raw_path / "avis_expertise_ASNR_2026.pdf"
)

# 2026 Board opinion
urllib.request.urlretrieve(
    "https://reglementation-controle.asnr.fr/content/download/209769/file/2026-AV-027.pdf",
    raw_path / "avis_ASNR_2026.pdf"
)

(WindowsPath('data/raw/avis_ASNR_2026.pdf'),
 <http.client.HTTPMessage at 0x1fb06661490>)

We will now read these PDFs         
[pypdf documentation on text extraction](https://pypdf.readthedocs.io/en/stable/user/extract-text.html)

We will use the example from the documentation but take all the pages and store them in a list. For each page, this list will contain a dictionary with the filename, the page number, and the text. We created a function to do it so we could reuse it.

In [76]:
def register_pdf(folder : Path)-> list | None: 
    """
    Reads all PDFs in the folder and returns a list where each page is a dictionary 
    containing the original filename, the page number, and the text.
    folder : path
    """
    pages = []
    for path in sorted(folder.glob("*.pdf")):
        try:
            reader = PdfReader(path)
            for i, page in enumerate(reader.pages, start=1): # enumerate take the number and the page
                pages.append({"origin": path.name, "page": i, "text": page.extract_text() or ""})
        except Exception as e:
            print(f"Error for {path.name} : {e}")
    return pages 

Now that we have created this function, we can convert it into a table. Each dictionary will become a row in the table.

In [77]:
df_pages = pd.DataFrame(register_pdf(raw_path))

print(df_pages.head(1))

               origin  page                                               text
0  avis_ASNR_2026.pdf     1  RÉPUBLIQUE FRANÇAISE \n \n \n \n 1/4 \n \n \nA...


It worked, we can clearly see the file name, the page number, and the beginning of the text. Pandas truncates the text because it is too long.

We will therefore check the text by displaying a full page.

In [78]:
print(df_pages.loc[0, "text"][:5000])

RÉPUBLIQUE FRANÇAISE 
 
 
 
 1/4 
 
 
Avis n° 2026-AV-027 de l’Autorité de sûreté nucléaire et de radioprotection  
du 19 mai 2026 relatif aux options de sûreté présentées par CALOGENA 
concernant sa démarche de diversification du système d’arrêt du réacteur CAL-30 
 
 
L’Autorité de sûreté nucléaire et de radioprotection,  
Vu le code de l’environnement, notamment le titre IX de son livre V et son article R. 593-14 ;   
Vu le code de la santé publique, notamment son article L. 1333-2 ;  
Vu l’arrêté du 7 février 2012 modifié fixant les règles générales relatives aux installations nucléaires de base ;  
Vu la décision n° 2015-DC-0532 de l’Autorité de sûreté nucléaire du 17 novembre 2015 relative au rapport de 
sûreté des installations nucléaires de base ; 
Vu le guide de l’Autorité de sûreté nucléaire n° 22 du 18 juillet 2017 relatif à la conception des réacteurs à eau 
sous pression ; 
Vu le courrier de CALOGENA référenc é CAL-030-L-0-GEN-LEE-001-01 du 31 octobre 2024, modifié par le 

We can therefore observe:
- line breaks in the middle of sentences
- words split in two
- double spaces
- "Vu..." referring to legal citations, which can clutter the explanations regarding the reactor

We therefore need to:
- replace line breaks with spaces
- remove double spaces
- ignore split words, as this is a minor issue

Now, let us consider how to reconstruct the broken lines while preserving the paragraphs. We need to compare more pages, let's look at some others.

In [79]:
print(df_pages.loc[5, "text"][:5000])

 
  
AVIS D’EXPERTISE N° 2026-00015 DU 02 FÉVRIER 2026 
  
2/6 
 
Le refroidissement du cœur repose sur la circulation naturelle de l’eau primaire pour toutes les situations du 
réacteur (fonctionnement normal, incidents et accidents) : l’eau chauffée par le cœur s’élève dans une colonne de 
grande hauteur , appelée cheminée centrale , puis redescend au travers de  l’échangeur primaire/secondaire 
implanté autour de la cheminée pour retourner dans le cœur. 
1.2. SYSTÈMES DE MAÎTRISE DE LA RÉACTIVITÉ EN SITUATIONS INCIDENTELLES ET ACCIDENTELLES 
La maîtrise de la réactivité du réacteur CAL -30 en situations incidentelles et accidentelles repose sur l’insertion 
automatique de grappes de crayons d’absorbants neutroniques dans le cœur, ces dernières étant réparties en un 
groupe de régulation et deux groupes d’arrêt. Le réacteur est équipé de deux systèmes d’arrêt d’urgence (AU), le 
premier, appelé « système d’AU principal », provoquant la chute gravitaire d es grappes du groupe de régul

In [80]:
print(df_pages.loc[8, "text"][:5000])

 
  
AVIS D’EXPERTISE N° 2026-00015 DU 02 FÉVRIER 2026 
  
5/6 
 
2.2.4. Enjeux associés à la défaillance de chute de grappes par mode commun 
Afin d’apprécier les enjeux de sûreté, la Direction de l’expertise en sûreté a demandé à la société CALOGENA, 
au cours de l’expertise, de fournir des éléments permettant d’apprécier les conséquences de la défaillance de la 
chute de toutes les grappes. La société CALOGENA a alors transmis des éléments visant à justifier  l’existence 
d’un délai significatif avant que le refroidissement du cœur du réacteur ne soit compromis. Dans le cas où l’échec 
de la chute des grappes proviendrait de défaillances d es systèmes de contrôle-commande ou des dispositifs de 
coupure de l’alimentation électrique des mécanismes, elle a alors précisé que des actions en local pourraient être 
menées dans ce délai en vue de  provoquer la chute des grappes par coupure forcée de l’alimentation électrique 
des mécanismes. L’ensemble de ces éléments préliminaires devra êt

We can observe that:
- the header is repeated on every page
- semicolons mark the end of legal paragraphs
- paragraphs often end with a period followed by more than three spaces

We will test our hypothesis regarding paragraphs marked by a period followed by more than three spaces across all documents.

In [81]:
counter = Counter() 
for text in df_pages["text"]:
    for line in text.split("\n"):
        paragraph = re.search(r"\. {3,}$", line)
        if paragraph:
            counter["3 spaces or more"] += 1 
counter

Counter({'3 spaces or more': 6})

There are only 6 examples of this pattern, a period followed by at least three spaces, so that is too few.

Instead, we will use pypdf's layout mode. This way, we will have the vertical spacing that marks the paragraphs.
Let us therefore evolve our function to incorporate this mode.

In [82]:
def register_pdf(folder : Path, mode : str)-> list | None: 
    """
    Reads all PDFs in the folder and returns a list where each page is a dictionary 
    containing the original filename, the page number, and the text.
    folder : path
    """
    pages = []
    for path in sorted(folder.glob("*.pdf")):
        try:
            reader = PdfReader(path)
            for i, page in enumerate(reader.pages, start=1): 
                text = page.extract_text(extraction_mode=mode)
                pages.append({"origin": path.name, "page": i, "text": text or ""})
        except Exception as e:
            print(f"Error for {path.name} : {e}")
    return pages 

In [83]:
df_pages = pd.DataFrame(register_pdf(raw_path, mode="layout"))
print(df_pages.head(1))
print(df_pages.loc[0, "text"][:5000])

               origin  page                                               text
0  avis_ASNR_2026.pdf     1                                                ...
                                                                            RÉPUBLIQUE FRANÇAISE

















         Avis n° 2026-                                                 AV-027                                                         de l’Autorité de sûreté nucléaire et de radioprotection
            du 19 mai 2026  relatif  aux options  de sûreté  présentées par CALOGENA
concernant sa démarche  de  diversification  du  système d’arrêt du  réacteur CAL-30



L’Autorité de s ûreté nucléaire                     et de radiopr otection,

Vu le code de l’env ironnement, notamment le titre IX de son livre V et son article R. 593-                    14 ;

Vu le code de la santé publique, notamment son article L. 1333-                                                          2 ;

Vu l’arrêté du 7 février 2012 modifié fixant 

In [84]:
print(df_pages.loc[5, "text"][:5000])

Le  refroidissement  du  cœur  repose  sur  la  circulation  naturelle  de  l’eau  primaire  pour  toutes  les  situations  du
réacteur (fonctionnement normal, incidents et accidents) : l’eau chauffée par le cœur s’élève dans une colonne de
grande  hauteur,     appelée     cheminée  centrale,       puis  redescend      au  travers  de    l’échangeur  primaire/secondaire
implanté autour de la cheminée pour retourner dans le cœur.
1.2.     SYSTÈMES DE MAÎTRISE DE LA RÉACTIVITÉ EN SITUATIONS INCIDENTELLES ET ACCIDENTELLES
La maîtrise de la réactivité du réacteur CAL-30 en situations incidentelles et accidentelles                     repose sur l’insertion
automatique de grappes de crayons d’absorbants neutroniques dans le cœur, ces dernières étant réparties en un
groupe de régulation et deux groupes d’arrêt. Le réacteur est équipé de deux systèmes d’arrêt d’urgence (AU), le
premier, appelé « système d’AU principal », provoquant la chute gravitaire des grappes du groupe de régulation
et de

The two documents do not operate in the same way. The "layout" approach works better for the expert assessment, whereas the "plain text" format works better for the board's opinion. We will therefore read these two documents differently.

Let's update the function to select a single element, and take the opportunity to set a default mode.

In [85]:
def register_pdf(path : Path, mode : str = "plain")-> list | None: 
    """
    Reads all PDFs in the folder and returns a list where each page is a dictionary 
    containing the original filename, the page number, and the text.
    folder : path
    mode : str with default mode "plain"
    """
    pages = []
    try:
        reader = PdfReader(path)
        for i, page in enumerate(reader.pages, start=1): 
            text = page.extract_text(extraction_mode=mode)
            pages.append({"origin": path.name, "page": i, "text": text or ""})
    except Exception as e:
        print(f"Error for {path.name} : {e}")
    return pages 

In [86]:
pages = (
    register_pdf(raw_path / "avis_expertise_ASNR_2026.pdf", mode="layout") +
    register_pdf(raw_path / "avis_ASNR_2026.pdf", mode="plain")
)

df_pages = pd.DataFrame(pages)
print(df_pages.loc[0, "text"][:5000])

                                                                  Monsieur le Directeur de la Direction des réacteurs innovants


                                                                                                Fontenay-aux-Roses, le 2 février 2026

           AVIS D’EXPERTISE N° 2026-00015 DU 02 FÉVRIER 2026

Objet :                CALOGENA – Projet de réacteur CAL-30 - Démarche de diversification pour les systèmes
                       assurant   la   maîtrise   de   la   réactivité   du   réacteur   en   situations   incidentelles   et
                       accidentelles.
Références :           [1] Saisine ASNR - CODEP-DRI-2025-028912 du 5 mai 2025.
                       [2] Arrêté du 7 février 2012 fixant les règles générales relatives aux installations nucléaires de
                           base.
                       [3] Guide ASN n° 22 : Conception des réacteurs à eau sous pression.

Dans  le  cadre  du  développement  de  son               projet  de  réact

In [87]:
print(df_pages.loc[5, "text"][:5000])

                                                                 ANNEXE
                            Recommandation de la Direction de l’expertise en sûreté



La Direction de l’expertise en sûreté recommande que, afin d’assurer, avec un haut niveau de fiabilité, la fonction
de  sûreté  liée  à  la  maîtrise  de  la  réactivité,  la  société  CALOGENA  prévoie              un  moyen  complémentaire  aux
dispositions prévues pour la fonction d’arrêt d’urgence, qui soit fonctionnellement diversifié.
































































































                                       AVIS D’EXPERTISE N° 2026-00015 DU 02 FÉVRIER 2026                                                6/6




Right, we have a clearer picture of the issues we still need to resolve:
- excessive spacing
- removing headers and footers
- removing multiple blank lines
- joining lines belonging to the same paragraph
- splitting paragraphs

Before that, let's run a few checks. Before that, let's perform a few checks. First, let's verify whether all the tags contain text. We can take this opportunity to gather information about the text length in a column, that will help us later on.

In [88]:
df_pages["text_len"] = df_pages["text"].str.len()
print(df_pages["text_len"])

0    3899
1    4930
2    4189
3    5151
4    4656
5     739
6    2548
7    3523
8    3922
9     584
Name: text_len, dtype: int64


Right, all our pages are present.

Let's move on to the cleaning function.     
First, we will:             
- remove sequences of spaces
- remove spaces around each line caused by the PDF
- remove empty lines (starting with a minimum of three line breaks)

In [89]:
def clean_text(text : str) -> str: 
    """
    Text cleaning function. It performs the following:
    - remove sequences of spaces and tabulation
    - remove spaces around each line caused by the PDF
    - remove empty lines (starting with a minimum of three line breaks)
    text : str
    """
    # security check 
    if not isinstance(text, str):
        return ""
    
    # remove sequences of spaces and tabulation
    text = re.sub(r"[ \t]+", " ", text)

    # remove spaces around each line caused by the PDF
    lines = text.split("\n")
    clean_lines = []
    
    for line in lines:
        clean_lines.append(line.strip())
    text = "\n".join(clean_lines)

    # remove empty lines (starting with a minimum of three line breaks)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

We have our cleaning function. We are going to create a new column with the cleaned text.

In [90]:
df_pages["clean"] = df_pages["text"].apply(clean_text)

df_pages["clean"][0]

'Monsieur le Directeur de la Direction des réacteurs innovants\n\nFontenay-aux-Roses, le 2 février 2026\n\nAVIS D’EXPERTISE N° 2026-00015 DU 02 FÉVRIER 2026\n\nObjet : CALOGENA – Projet de réacteur CAL-30 - Démarche de diversification pour les systèmes\nassurant la maîtrise de la réactivité du réacteur en situations incidentelles et\naccidentelles.\nRéférences : [1] Saisine ASNR - CODEP-DRI-2025-028912 du 5 mai 2025.\n[2] Arrêté du 7 février 2012 fixant les règles générales relatives aux installations nucléaires de\nbase.\n[3] Guide ASN n° 22 : Conception des réacteurs à eau sous pression.\n\nDans le cadre du développement de son projet de réacteur nucléaire calogène, appelé CAL-30, la société\nCALOGENA a sollicité l’avis de l’Autorité de sûreté nucléaire et de radioprotection (ASNR) sur l’approche mise en\nœuvre, au stade des options de sûreté, pour la diversification du système d’arrêt du réacteur.\nDans ce contexte, par la lettre citée en première référence, la Direction des réacteu

So, certain sentences that are split across multiple lines. We will therefore retain only paragraphs as the primary separators, line breaks that do not constitute paragraphs will be removed.

In [91]:
def clean_text(text : str) -> str: 
    """
    Text cleaning function. It performs the following:
    - remove sequences of spaces and tabulation
    - remove spaces around each line caused by the PDF
    - remove empty lines (starting with a minimum of three line breaks)
    - cut the paragraphs and join the cut lines
    text : str
    """
    # security check 
    if not isinstance(text, str):
        return ""
    
    # remove sequences of spaces and tabulation
    text = re.sub(r"[ \t]+", " ", text)

    # remove spaces around each line caused by the PDF
    lines = text.split("\n")
    clean_lines = []

    for line in lines:
        clean_lines.append(line.strip())
    text = "\n".join(clean_lines)

    # remove empty lines (starting with a minimum of three line breaks)
    text = re.sub(r"\n{3,}", "\n\n", text)

    # cut the paragraphs and join the cut lines
    joined_lines = []

    for line in text.split("\n"):
        precedent_line = joined_lines[-1] if joined_lines else "" # We take the previous sentence, provided it is not empty
        sentence = precedent_line != "" and not precedent_line.endswith((".", ";", ":")) # The sentence includes the preceding line if it is not interrupted by punctuation
        
        if sentence and line != "":
            joined_lines[-1] += " " + line # The sentence will be the two lines joined by a space
        else:
            joined_lines.append(line) # If it is not a continuation of a sentence, the new line is added as the start of a new sentence
    text = "\n".join(joined_lines)

    return text.strip()

The function is ready, we can try it out on our text.

In [92]:
df_pages["clean"] = df_pages["text"].apply(clean_text)

print(df_pages.loc[1, "clean"][:5000])

Le refroidissement du cœur repose sur la circulation naturelle de l’eau primaire pour toutes les situations du réacteur (fonctionnement normal, incidents et accidents) : l’eau chauffée par le cœur s’élève dans une colonne de grande hauteur, appelée cheminée centrale, puis redescend au travers de l’échangeur primaire/secondaire implanté autour de la cheminée pour retourner dans le cœur.
1.2. SYSTÈMES DE MAÎTRISE DE LA RÉACTIVITÉ EN SITUATIONS INCIDENTELLES ET ACCIDENTELLES La maîtrise de la réactivité du réacteur CAL-30 en situations incidentelles et accidentelles repose sur l’insertion automatique de grappes de crayons d’absorbants neutroniques dans le cœur, ces dernières étant réparties en un groupe de régulation et deux groupes d’arrêt. Le réacteur est équipé de deux systèmes d’arrêt d’urgence (AU), le premier, appelé « système d’AU principal », provoquant la chute gravitaire des grappes du groupe de régulation et de l’un des groupes d’arrêt, le second, appelé « système d’AU diversif

It worked, each sentence is on a single line, and the paragraphs are separated by line breaks.

In [93]:
print(df_pages.loc[7, "clean"][:5000])

Avis n° 2026-AV-027 de l’ASNR du 19 mai 2026 2/4

Rend l’avis suivant :
Le réacteur CAL-30 de CALOGENA est un réacteur nucléaire d’une puissance de 30 MW thermiques utilisant de l’eau légère à la fois comme modérateur et comme caloporteur.
Pour assurer la maîtrise de sa réactivité, ce réacteur dispose de deux systèmes d’arrêt d’urgence (AU), le premier qualifié de « principal » et le second de « diversifié », chacun permettant , en situation incidentelle ou accidentelle, d’atteindre rapidement un état sous-critique du cœur.
Ces deux systèmes reposent sur un même principe consistant en une insertion gravitaire rapide d’un ensemble de grappes de crayons d’absorbants neutroniques dans le cœur.

La sollicitation de ces deux systèmes d’arrêt d’urgence du réacteur CAL-30 est induite selon la même séquence suivante :
• étape 1 : détection d’une situation anormale nécessitant l’arrêt du réacteur et émission du signal d’arrêt d’urgence ;
• étape 2 : ouverture des dispositifs de préhension des t

We will now look at sentences that are split across pages.
The idea is as follows: if the last sentence ends with a period, it marks the end of a line, otherwise the sentences need to be reassembled across the documents. The challenge is the header that repeats at the start of each document, so it must be ignored.

In [94]:
def boolean_end_page_line(text : str) -> bool:
    """
    Function that returns True if the last line of the page continues onto the next page.
    text : str
    """
    for line in reversed(text.split("\n")): # we take the last line of the page
        if line == "" or line.startswith(("AVIS", "Adresse", "Tél")) or line[0].isdigit(): # if it's a stray line
            continue 
        return not line.endswith((".", ";", ":")) # We want True if the sentence is broken. `endswith` returns True, so we reverse it with `not`
    return False # If there is no line of text

Let's now look at which page is cut off and record this information in a column.

In [95]:
df_pages["end_page_line_cut"] = df_pages["clean"].apply(boolean_end_page_line)
print(df_pages[["origin", "page", "end_page_line_cut"]])

                         origin  page  end_page_line_cut
0  avis_expertise_ASNR_2026.pdf     1              False
1  avis_expertise_ASNR_2026.pdf     2               True
2  avis_expertise_ASNR_2026.pdf     3              False
3  avis_expertise_ASNR_2026.pdf     4              False
4  avis_expertise_ASNR_2026.pdf     5               True
5  avis_expertise_ASNR_2026.pdf     6              False
6            avis_ASNR_2026.pdf     1               True
7            avis_ASNR_2026.pdf     2              False
8            avis_ASNR_2026.pdf     3              False
9            avis_ASNR_2026.pdf     4              False


Let's look at an example of these cuts.

In [96]:
print("Ending :", df_pages.loc[1, "clean"][-500:])
print("Beginning :", df_pages.loc[2, "clean"][:500])

Ending : e commande de chaque système d’AU, leur conception étant différente. S’agissant de l’alimentation électrique associée, chaque mécanisme dispose de sa propre boucle

2 Le domaine de conception étendu désigne l’ensemble des situations, retenues pour la conception d’une installation nucléaire, plus sévères que celles considérées dans le domaine de conception de référence. Il recouvre notamment les situations résultant de défaillances multiples.

AVIS D’EXPERTISE N° 2026-00015 DU 02 FÉVRIER 2026 2/6
Beginning : de courant. Le choix d’une diversification du matériel utilisé par chaque système d’AU pour couper l’alimentation électrique des mécanismes (disjoncteurs notamment) n’est pas arrêté à ce stade.

La société CALOGENA ne prévoit pas de diversifier les équipages mobiles et leurs structures de guidage (interne supérieur et tubes guides des AC). Elle identifie la déformation mécanique importante des structures dans lesquelles les grappes et les tiges de commande coulissent comme 

"de sa propre boucle""de courant."  are indeed two parts of the same phrase, our filter therefore worked.

To connect them, the simplest approach is first to remove the thin parasitic strands.

In [97]:
def remove_parasite_page_end(text : str) -> str:
    """
    Removes page endings that do not constitute meaningful lines.
    text : str
    """
    lines = text.split("\n")
    # as long as the line is either empty, starts with a stray prefix, or is a page number, delete it
    while lines and (lines[-1] == "" or lines[-1].startswith(("AVIS", "Adresse", "Tél")) or lines[-1][0].isdigit()):
        lines.pop() 
    return "\n".join(lines)

We test the function.

In [98]:
for i in df_pages.index:
    if df_pages.loc[i, "end_page_line_cut"]:
        df_pages.loc[i, "clean"] = remove_parasite_page_end(df_pages.loc[i, "clean"])

print(df_pages.loc[1, 'clean'][-500:])

stème d’AU principal. S’agissant de l’instrumentation, la démarche se fonde sur la mesure de paramètres physiques différents. En outre, la diversification des systèmes de contrôle-commande des deux systèmes d’AU repose sur le choix de technologies et/ou de fournisseurs différents.

La diversification concerne également les mécanismes de commande de chaque système d’AU, leur conception étant différente. S’agissant de l’alimentation électrique associée, chaque mécanisme dispose de sa propre boucle


Right, now we can use this function to combine all the pages into a single text per source. We therefore remove the extraneous endings from all the pages.

In [99]:
df_pages["clean"] = df_pages["clean"].apply(remove_parasite_page_end)

# test on the second pdf
print(df_pages.loc[6, "clean"][-500:])
print(df_pages.loc[7, "clean"][:500])


 pour les réacteurs innovants référencé CODEP-DFP-2026-017964 du 19 mars 2026, ;
Considérant que, par courrier du 31 octobre 2024 susvisé, modifié par le courrier du 19 juin 2025 susvisé, CALOGENA a sollicité l’avis de l’Autorité de sûreté nucléaire et de radioprotection sur les options de sûreté relatives à la diversification du système d’arrêt de son réacteur CAL- 30 définies par les documents CAL-030- E-0-SUR-DTE-001 et CAL-030-E-1-SUR-NTE-037 dans leur versions mises à jour au 14 avril 2026,
Avis n° 2026-AV-027 de l’ASNR du 19 mai 2026 2/4

Rend l’avis suivant :
Le réacteur CAL-30 de CALOGENA est un réacteur nucléaire d’une puissance de 30 MW thermiques utilisant de l’eau légère à la fois comme modérateur et comme caloporteur.
Pour assurer la maîtrise de sa réactivité, ce réacteur dispose de deux systèmes d’arrêt d’urgence (AU), le premier qualifié de « principal » et le second de « diversifié », chacun permettant , en situation incidentelle ou accidentelle, d’atteindre rapidement 

You can see that the second document uses "Avis" instead of the "AVIS" found in the first; consequently, it was not picked up by the split process, so the function needs to be updated. Also, these parasites are found at the beginning of the document. Let's look at another pages of the second one before evolving the function.

In [100]:
print("Start of page 9 : ", df_pages.loc[8, "clean"][:500])
print("End of page 9 :", df_pages.loc[8, "clean"][-500:])

print("Start of page 10 :",df_pages.loc[9, "clean"][:500])
print("End of page 10 : ",df_pages.loc[9, "clean"][-500:])

Start of page 9 :  Avis n° 2026-AV-027 de l’ASNR du 19 mai 2026 3/4

2. Absence de diversification fonctionnelle des parties mobiles des systèmes d’arrêt d’urgence sollicitées à l’étape 3 Concernant la fiabilité de la chute gravitaire des barres libérées de leur système de préhension, CALOGENA retient, pour les parties mobiles (grappes et leurs tiges de commande, ces dernières étant particulièrement longues) et leurs dispositifs de guidage, des exigences de conception visant à prévenir toute déformation mécanique 
End of page 9 :  et robuste du réacteur est à prévoir en dernier recours afin d’être toujours en capacité d’arrêter la réaction nucléaire.
L’ASNR considère que, dans le cadre d’une éventuelle demande d’autorisation de création, CALOGENA devra présenter :
- un moyen alternatif2, simple et robuste, d’arrêt de la réaction en chaîne en cas de blocage total des mécanismes des grappes. CALOGENA devra préciser si ce moyen relève de la démonstration de sûreté de son réacteur ou s’il 

Let's look at another page to get an exhaustive inventory of the unwanted headers and footers.

In [101]:
print("Start of page 9 :",df_pages.loc[8, "clean"][:1000])
print("End of page 9 : ",df_pages.loc[8, "clean"][-1000:])

Start of page 9 : Avis n° 2026-AV-027 de l’ASNR du 19 mai 2026 3/4

2. Absence de diversification fonctionnelle des parties mobiles des systèmes d’arrêt d’urgence sollicitées à l’étape 3 Concernant la fiabilité de la chute gravitaire des barres libérées de leur système de préhension, CALOGENA retient, pour les parties mobiles (grappes et leurs tiges de commande, ces dernières étant particulièrement longues) et leurs dispositifs de guidage, des exigences de conception visant à prévenir toute déformation mécanique susceptible de conduire à un blocage mécanique des grappes. Pour le dimensionnement de ces composants, le cas de charge pénalisant retenu par CALOGENA correspond aux sollicitations mécaniques induites par des séismes extrêmes 1. En complément à cette exigence de dimensionnement, CALO GENA prévoit la réalisation d’essais de chute de grappes afin de vérifier l’absence de blocage mécanique malgré un désalignement forcé des structures de guidage, simulant de manière pénalisante les

So, compared to Document 1, we can see that Document 2 additionally includes:           

in the header:          
- "Avis"             
       
in the footer:          
- "2 le domaine de conception": the line starts with a number and might be picked up by our filter
- "Adresse" and "Tél.", just like in Document 1

Let's revisit our function and evolve it accordingly.

In [102]:
def remove_parasite(text : str) -> str:
    """
    Removes unwanted headers and footers.
    text : str
    """
    # security 
    if not isinstance(text, str):
        return ""
    
    lines = text.split("\n")
    # bottom of the page 
    # as long as the line is either empty, starts with a stray prefix, or is a page number, delete it
    while (
        lines 
        and (lines[-1] == "" 
            or lines[-1].startswith(("AVIS", "Avis", "Adresse", "Tél")) 
            or (lines[-1][0].isdigit() and len(lines[-1])<10)
        )
    ):
        lines.pop() 

    # in the lead of the page 
    while (
        lines 
        and (lines[0] == "" 
            or lines[0].startswith(("AVIS", "Avis", "Adresse", "Tél")) 
            or (lines[0][0].isdigit() and len(lines[0])<10)
        )
    ):
        lines.pop(0)
    
    return "\n".join(lines)

There's still the issue of the explanatory footnote. If we remove it, we'll lose important information. Since, for now, we don't have two consecutive pages with sentences cut off at the end of each one, we'll move it to the bottom of the next page. 

In [103]:
def delete_footnote(text : str) -> tuple: 
    """
    delete and save the footnote to the bottom of the selected page
    text : str : the text on the page 
    selected_page_for_retrieval : int : the page that will retrieve the footnote. 1 means the next one 
    """
    lines = text.split("\n")

    footnote_list = []
    while lines and len(lines[-1])> 7 and lines[-1][0].isdigit() and lines[-1][1] == " ":
            line = lines.pop()
            footnote_list.insert(0, line)

    # return the cleaned text and the deleted footnote  
    return "\n".join(lines).strip(), "\n".join(footnote_list) # strip() because there is a empty line before a footnote
        


The function that removes the footnote is ready. Now we'll make sure that this footnote isn't lost: we'll add it to the next page 

In [104]:
def remove_footnote(df : pd.DataFrame, selected_page : int = 1, indicator_column : str = "end_page_line_cut", clean_column : str = "clean") -> None:
    """
    Use the `delete_footnote` function to delete the footnote and move it to another page.
    df = pd.DataFrame 
    selected_page : int. the next page where the note will be pasted. 1 means “the page immediately following”
    indicator_column : str. Column in the dataframe containing the boolean values indicating which pages need to be corrected
    clean_column : str. Column where the cleaned text will be stored
    """
    for i in df.index:
        if df.loc[i, indicator_column]:
            cleaned_text, footnote = delete_footnote(df.loc[i, clean_column])
            df.loc[i, clean_column] = cleaned_text

            if footnote != "":
                next_page = i + selected_page 
                next_page_text = df.loc[next_page, clean_column]
                df.loc[next_page, clean_column] = next_page_text + "\n" + footnote

In [105]:
print(df_pages[["origin", "page", "end_page_line_cut"]])

                         origin  page  end_page_line_cut
0  avis_expertise_ASNR_2026.pdf     1              False
1  avis_expertise_ASNR_2026.pdf     2               True
2  avis_expertise_ASNR_2026.pdf     3              False
3  avis_expertise_ASNR_2026.pdf     4              False
4  avis_expertise_ASNR_2026.pdf     5               True
5  avis_expertise_ASNR_2026.pdf     6              False
6            avis_ASNR_2026.pdf     1               True
7            avis_ASNR_2026.pdf     2              False
8            avis_ASNR_2026.pdf     3              False
9            avis_ASNR_2026.pdf     4              False


In [106]:
remove_footnote(df_pages)

print("END OF PAGE 2 : ", df_pages.loc[1, "clean"][-4000:])
print("END OF PAGE 3 : ", df_pages.loc[2, "clean"][-4000:])

END OF PAGE 2 :  rande hauteur, appelée cheminée centrale, puis redescend au travers de l’échangeur primaire/secondaire implanté autour de la cheminée pour retourner dans le cœur.
1.2. SYSTÈMES DE MAÎTRISE DE LA RÉACTIVITÉ EN SITUATIONS INCIDENTELLES ET ACCIDENTELLES La maîtrise de la réactivité du réacteur CAL-30 en situations incidentelles et accidentelles repose sur l’insertion automatique de grappes de crayons d’absorbants neutroniques dans le cœur, ces dernières étant réparties en un groupe de régulation et deux groupes d’arrêt. Le réacteur est équipé de deux systèmes d’arrêt d’urgence (AU), le premier, appelé « système d’AU principal », provoquant la chute gravitaire des grappes du groupe de régulation et de l’un des groupes d’arrêt, le second, appelé « système d’AU diversifié », provoquant la chute gravitaire de l’autre groupe d’arrêt. En outre, l’activation de l’un des systèmes entraîne la chute des grappes activée par l’autre système (option retenue à ce stade par la société C

Great, the footnote has been moved to the bottom of page 3, which is no longer a problem since it doesn't have a line break. 

Let's add a safety measure: if the next page also has a truncated line, we move on to the page after that. 

In [107]:
def remove_footnote(df : pd.DataFrame, indicator_column : str = "end_page_line_cut", clean_column : str = "clean") -> None:
    """
    Use the `delete_footnote` function to delete the footnote and move it to another page that doesn't have a line break at the end.
    indicator_column : str. Column in the dataframe containing the boolean values indicating which pages need to be corrected
    clean_column : str. Column where the cleaned text will be stored
    """
    for i in df.index:
        if df.loc[i, indicator_column]:
            cleaned_text, footnote = delete_footnote(df.loc[i, clean_column])
            df.loc[i, clean_column] = cleaned_text

            if footnote != "":
                next_page = i + 1 
                while df.loc[next_page, indicator_column]:
                    next_page += 1 

                next_page_text = df.loc[next_page, clean_column]
                df.loc[next_page, clean_column] = next_page_text + "\n" + footnote

Now we realize that the `remove_parasite_page_end` function has removed the notes at the end of the pages. So we'll set it aside and use the `remove_parasite` function to reconstruct the cleaned up text and apply our new function.

In [108]:
df_pages["clean"] = df_pages["text"].apply(clean_text)
df_pages["clean"] = df_pages["clean"].apply(remove_parasite)
df_pages["end_page_line_cut"] = df_pages["clean"].apply(boolean_end_page_line)
remove_footnote(df_pages)

Let's check

In [109]:
print("END OF PAGE 2 : ", df_pages.loc[1, "clean"][-4000:])
print("END OF PAGE 3 : ", df_pages.loc[2, "clean"][-4000:])

END OF PAGE 2 :  rande hauteur, appelée cheminée centrale, puis redescend au travers de l’échangeur primaire/secondaire implanté autour de la cheminée pour retourner dans le cœur.
1.2. SYSTÈMES DE MAÎTRISE DE LA RÉACTIVITÉ EN SITUATIONS INCIDENTELLES ET ACCIDENTELLES La maîtrise de la réactivité du réacteur CAL-30 en situations incidentelles et accidentelles repose sur l’insertion automatique de grappes de crayons d’absorbants neutroniques dans le cœur, ces dernières étant réparties en un groupe de régulation et deux groupes d’arrêt. Le réacteur est équipé de deux systèmes d’arrêt d’urgence (AU), le premier, appelé « système d’AU principal », provoquant la chute gravitaire des grappes du groupe de régulation et de l’un des groupes d’arrêt, le second, appelé « système d’AU diversifié », provoquant la chute gravitaire de l’autre groupe d’arrêt. En outre, l’activation de l’un des systèmes entraîne la chute des grappes activée par l’autre système (option retenue à ce stade par la société C

All right, so we've reconstructed a clean version of the text and moved the endnotes to the right places.

Now we need to think about what comes next: we want an assistant that specifies exactly which page the information is on, otherwise it’s not really useful. 

In that case, what should we do with lines that span two pages?  

We can: 
- create a row in the table specifically for this situation. It will contain only the split paragraph, with the notation “page x and y” in the column corresponding to the page number in question. However, this will isolate it from its context and therefore result in a poor embedding. 

- Second solution: Using a simple `len()` function, we can determine the number of characters on a page and use that to pinpoint exactly where a quote is located. 

In [138]:
def pdf_reconstruction(df : pd.DataFrame) -> dict : 
    """
    Groups the pages of the documents while preserving the associated page numbers. 
    Returns the result as a dictionary.
    df : pd.DataFrame
    """
    documents = {}
    for origin, df_documents in df.groupby("origin"):

        starting_page = []
        text = ""

        for i in df_documents.index:
            starting_page.append((len(text), int(df_documents.loc[i, "page"]))) # "int" because pandas save in np.int64

            if df_documents.loc[i, "end_page_line_cut"]:
                separation = " "
            else:
                separation = "\n"
            
            text += df_documents.loc[i, "clean"] + separation
        documents[origin] = (text.strip(), starting_page)

    return documents 

In [139]:
documents = pdf_reconstruction(df_pages)
print(documents.keys())

dict_keys(['avis_ASNR_2026.pdf', 'avis_expertise_ASNR_2026.pdf', 'dossier_concertation_calogena_2026.pdf'])


All right, our function has successfully sorted the two documents into separate groups. Now let's take a closer look at this.

In [140]:
text, starting_page = documents["avis_expertise_ASNR_2026.pdf"]
print(starting_page)

[(0, 1), (3056, 2), (7283, 3), (11319, 4), (15933, 5), (20070, 6)]


Our code has correctly numbered the pages based on the length of the text.

In [141]:
page_03 = starting_page[2][0]
print("END OF PAGE 2 :", repr(text[page_03 - 200 : page_03]))
print("START OF PAGE 3 :", repr(text[page_03 : page_03 + 200]))

END OF PAGE 2 : ' concerne également les mécanismes de commande de chaque système d’AU, leur conception étant différente. S’agissant de l’alimentation électrique associée, chaque mécanisme dispose de sa propre boucle '
START OF PAGE 3 : 'de courant. Le choix d’une diversification du matériel utilisé par chaque système d’AU pour couper l’alimentation électrique des mécanismes (disjoncteurs notamment) n’est pas arrêté à ce stade.\n\nLa so'


In [142]:
full_text, page_starts = documents["avis_ASNR_2026.pdf"]
print(page_starts)

start_page_2 = page_starts[1][0]
print("END OF PAGE 1 :", repr(full_text[start_page_2 - 200 : start_page_2]))
print("START OF PAGE 2 :", repr(full_text[start_page_2 : start_page_2 + 200]))

[(0, 1), (2493, 2), (5884, 3), (9686, 4)]
END OF PAGE 1 : 'atives à la diversification du système d’arrêt de son réacteur CAL- 30 définies par les documents CAL-030- E-0-SUR-DTE-001 et CAL-030-E-1-SUR-NTE-037 dans leur versions mises à jour au 14 avril 2026, '
START OF PAGE 2 : 'Rend l’avis suivant :\nLe réacteur CAL-30 de CALOGENA est un réacteur nucléaire d’une puissance de 30 MW thermiques utilisant de l’eau légère à la fois comme modérateur et comme caloporteur.\nPour assur'


We can see that the page join worked well. Also, the endnote is no longer at the end of page 2, as intended. That was the critical points of this assembly.

So, for now, we're done with the data engineering for these two PDFs. Let's move on to another document now. 

## 02. Calogena Consultation Report

We will now review the public consultation document published by Calogena. Its purpose is to inform the public prior to a decision being made. In this case, it concerns the company’s proposed thermal reactor project at the CEA site in Cadarache.

In [143]:
url = "https://www.calogena-cadarache-concertation.fr/media/d632c2e3587fa3cabc14/Calogena_Cadarache_Dossier_concertation_WEB.pdf"
request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

with urllib.request.urlopen(request) as response:
    (raw_path / "dossier_concertation_calogena_2026.pdf").write_bytes(response.read())

reader = PdfReader(raw_path / "dossier_concertation_calogena_2026.pdf")
print("pages :", len(reader.pages))

pages : 55


We've loaded the file, we can load it into a Pandas DataFrame.

In [144]:
consultation_path = raw_path / "dossier_concertation_calogena_2026.pdf"

df_consultation = pd.DataFrame(register_pdf(consultation_path, mode="plain"))
print(repr(df_consultation.iloc[10][:10000]))

Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


origin               dossier_concertation_calogena_2026.pdf
page                                                     11
text      CadaracheProjet de chaufferie nucléaire\n2026D...
Name: 10, dtype: object


We can see a limitation: too many graphics make the PDF difficult to read. We're going to isolate these pages.
To do this, we need to determine the length of the text on each page. A page with a lot of graphics will have an unusually short amount of text.

In [145]:
print(df_consultation.head(1))

                                   origin  page                        text
0  dossier_concertation_calogena_2026.pdf     1  Dossier  \nde concertation


In [146]:
df_consultation["text_len"] = df_consultation["text"].str.len()
df_consultation[["page", "text_len"]].sort_values("text_len")

,page,text_len
53,54,0
54,55,12
0,1,25
1,2,58
52,53,108
41,42,389
38,39,804
9,10,826
48,49,840
19,20,863


There is a clear distinction between pages with fewer than 400 characters and the others, which start at 800 characters. Page 5 seems unusually long. Let's take a closer look at these pages.

In [147]:
shortest_text = df_consultation.sort_values("text_len").head(6)

for text in shortest_text["text"]:
    print(repr(text[:1000]))

''
'calogena.com'
'Dossier  \nde concertation'
'Concertation préalable\ndu 28 septembre au 28 novembre 2026'
'Responsable de la publication\nJulien Dereux\nConception, rédaction et mise en pages\nSeptembre 2026 © Calogena'
'CadaracheProjet de chaufferie nucléaire\n2026Dossier de concertation\n82 83\nLES RETOMBÉES  \nÉCONOMIQUES\n8\n8.1 -  À l’échelle locale ............................................................................................................................................................. 84\n8.2 -  À l’échelle nationale et européenne.................................................... 84\n'


This page contains only the text that is displayed. It therefore has no value for our RAG, so we can disregard it. 

In [148]:
df_consultation = df_consultation[df_consultation["text_len"] >= 400].reset_index(drop=True)


Now let's look at the pages with fewer than 1,000 characters.

In [149]:
shortest_text_1000 = df_consultation[df_consultation["text_len"] < 1000]

for text in shortest_text_1000["text"]:
    print(repr(text[:1000]))

'CadaracheProjet de chaufferie nucléaire\n2026Dossier de concertation\n10 11\nINTRO- \nDUCTION \n1\n1.1 - Le projet en bref ......................................................................................................................................................................... 12\n Le projet en chiffres .............................................................................................................................................................................................. 14\n La localisation du centre .............................................................................................................................................................................. 15\n1.2 - Le porteur du projet ............................................................................................................................................... 16\n'
'CadaracheProjet de chaufferie nucléaire\n2026Dossier de concertation\n18 19\nLA CONCERTATION  \nPRÉA

These pages are for chapter titles and the table of contents. We don't need them, so we can delete them.

In [150]:
df_consultation = df_consultation[df_consultation["text_len"] >= 1000].reset_index(drop=True)

Now let's take a look at the under 2 000

In [151]:
shortest_text_2000 = df_consultation[df_consultation["text_len"] < 2000]

for text in shortest_text_2000["text"]:
    print(repr(text[:2000]))

'CadaracheProjet de chaufferie nucléaire\n2026Dossier de concertation\n22 23\nLA CONCERTATION  \nPRÉALABLE\n2\n2.6 - Comment s’informer et participer ?\nLa concertation préalable autour du projet de chaufferie nucléaire \nau centre CEA de Cadarache se déroule du 28 septembre 2026 au \n28 novembre 2026 inclus. Des formats variés sont proposés pour \ns’informer et échanger sur le projet.\nRéunions  \npubliques\nRéunion publique d’ouverture\n> Le 30 septembre à 18 h 30\nChâteau de Cadarache  \nRoute de Vinon-sur-Verdon  \n13115 Saint-Paul-lès-Durance \nRéunion publique de clôture\n> Le 25 novembre à 18 h 30\nSalle Osco Manosco\n570 chemin du Moulin-Neuf  \n04100 Manosque\nRencontres  \nde proximité\nManosque\nVillage des sciences\n> Le 10 octobre 2026, de 13 h à 19 h\nSalle Osco Manosco  \n570 chemin du Moulin-Neuf  \n04100 Manosque\nVinon-sur-Verdon \nMarché de Vinon-sur-Verdon\n> Le 11 octobre 2026, de 8 h 30 à 11 h \nWebinaires (en ligne)\nCaractéristiques, enjeux de développement \nde

These pages contain useful information, so we won't delete them. We will therefore stop deleting pages based on their length. 

Now we're going to do the same thing we did with PDFs: we'll look for instances of unwanted text and delete them. 

We're going to use the functions we've already created and see what text they produce. 

In [152]:
df_consultation["clean"] = df_consultation["text"].apply(clean_text)

Now we're going to look at the artifacts at the beginning and end of the page

In [153]:
start_page = Counter()
end_page = Counter() 

for text in df_consultation["clean"]:
    line = text.split("\n")
    start_page.update(line[:10])
    end_page.update(line[-10:])

print("The pattern that appears most frequently at the top of the page:", start_page.most_common(15))
print("The pattern that appears most frequently at the bottom of the page:", end_page.most_common(15))

The pattern that appears most frequently at the top of the page: [('CadaracheProjet de chaufferie nucléaire 2026Dossier de concertation 4 5 D ans ce contexte, Calogena a été créée pour mettre l’éner- gie nucléaire au service d’un usage concret, local et immédiatement utile : le chauffage urbain. Notre conviction est que la technologie nucléaire, forte de plu - sieurs décennies d’expérience, peut être dédiée, non pas à la production d’électricité, mais à celle de chaleur bas carbone, via une chaufferie nucléaire pilotable, compacte et sûre.', 1), ('Aujourd’hui, 80 millions d’Européens sont raccordés à un réseau de chauf - fage urbain, dont la moitié d’entre eux sont alimentés par du gaz ou du charbon.', 1), ('Avec le projet de chaufferie nucléaire sur le centre CEA de Cadarache, nous voulons franchir une étape décisive. Le centre CEA de Cadarache offre un cadre particulièrement pertinent : il s’agit d’un site de référence dans le domaine nucléaire, doté d’une culture de sûreté reconnue,

We can see that each page begins with the same header:          
"CadaracheProjet de chaufferie nucléaire            
2026Dossier de concertation             
4 5"        


Our `clean_text` function removed and pasted these lines because they don't end with a period. So we'll remove them from the original text.     
So we're going to create a function that removes that header from this report

In [154]:
def remove_false_start_consultation(text : str) -> str: 
    """
    Deletes the first three lines of the Colagena consultation report
    text : str. the text of one page 
    """
    false_start = r"^CadaracheProjet de chaufferie nucléaire\n2026Dossier de concertation\n\d+ \d+\n?"

    return re.sub(false_start, "", text)

In [155]:
print(repr(df_consultation.iloc[3]["text"][:120]))
print(repr(remove_false_start_consultation(df_consultation.iloc[3]["text"])[:500]))

'CadaracheProjet de chaufferie nucléaire\n2026Dossier de concertation\n12 13\n1.1 - Le projet en bref\nLe projet de chaufferi'
'1.1 - Le projet en bref\nLe projet de chaufferie nucléaire de Cadarache vise la production de chaleur  \nbas carbone. Aujourd’hui, environ un tiers de la chaleur urbaine en France est produit \npar de l’énergie fossile1, et près de la moitié en Europe. De manière plus globale, \nla majorité du chauffage est fossile2. Le développement d’une solution à partir \nd’énergie nucléaire permettrait d’offrir une alternative décarbonée, pilotable et \ncompacte pour distribuer de la chaleur en ville respectueuse'


Now that this has been removed, we can run the old cleaning functions on our text again.

In [156]:
df_consultation["clean"] = (
    df_consultation["text"]
    .apply(remove_false_start_consultation)
    .apply(clean_text)
    .apply(remove_parasite)
)

In some tables of contents, we can bypass the character count filter because of the dots that inflate the count. So we'll remove the pages with lines of dots to eliminate the tables of contents. 

In [157]:
summary = df_consultation["clean"].str.count(r"\.{5,}") >= 2 
df_consultation = df_consultation[~summary].reset_index(drop=True)

Now we're going to look at the pages where sentences are cut off mid-sentence and apply the filter to the footnotes.


In [158]:
df_consultation["end_page_line_cut"] = df_consultation["clean"].apply(boolean_end_page_line)
remove_footnote(df_consultation)

print(df_consultation["end_page_line_cut"].value_counts())

end_page_line_cut
False    25
True     14
Name: count, dtype: int64


In [159]:
print(df_consultation.iloc[5]["clean"][:20000])

LA CONCERTATION PRÉALABLE 2 2.1 - Le cadre réglementaire de la concertation préalable 2.3 - Les garants désignés par la CNDP 2.2 - La sollicitation volontaire de la CNDP La concertation préalable est une pro- cédure organisée en amont d’un projet susceptible d’avoir un impact sur l’en - vironnement, le cadre de vie ou l’acti - vité économique d’un territoire. Décrite aux articles L.121-15-1, L.121-16 et L.121-16-1 du Code de l’environnement, elle a notamment pour objectifs :
• d’informer le public de manière claire et transparente sur le projet, ses objectifs et ses caractéristiques ;
• de permettre à chacun de s’exprimer et de formuler des observations dès les premières phases du projet ;
• d’éclairer la décision du maître d’ouvrage grâce aux échanges et aux enseignements issus de la concertation.
Lors de sa séance plénière du 6 mai 2026, la CNDP a désigné MM. Jacques Finetti et Jean-Michel Fourniau, garants du processus de concerta - tion préalable pour le projet de chauf- ferie nucl

In [160]:
print(df_consultation.iloc[15]["clean"][:20000])

L’opportunité de développer l’appli - cation du nucléaire à des besoins de chaleur a été soulignée par différents rapports.
D’après Carbone 4 par exemple, dans son étude intitulée Les Small Modular Reactors (SMR), un levier important de décarbonation des réseaux de chaleur en Europe ? (2026)22, les solutions déjà déployées aujourd’hui pour la pro - duction de chaleur bas carbone (géo - thermie, solaire thermique, pompes à chaleur, chaleur fatale, biomasse) ne pourront couvrir au maximum que 75 % de la demande des réseaux de chaleur urbains à horizon 2050. Les 25 % res- tants pourraient être couverts par les SMR calogènes.
Dans son rapport sur l’intégration des SMR/AMR dans les systèmes énergé - tiques23, la Commission de régulation de l’énergie (CRE) précise que « les usages les plus prometteurs des SMR dans les pays européens dont l’électri- cité est déjà bas carbone résident dans la production de chaleur, notamment pour les réseaux de chaleur urbains et la chaleur industrielle inféri

In [161]:
print(df_consultation.iloc[30]["clean"][:20000])

Dans le cas du projet développé par Calogena, plusieurs caractéristiques de conception contribuent à cette approche :
• une architecture simplifiée, limitant le nombre d’équipements et d’interfaces susceptibles de contribuer à un dysfonctionnement ;
• un recours important aux systèmes passifs, capables d’assurer certaines fonctions de sûreté sans alimentation électrique ni intervention humaine immédiate ;
• un fonctionnement à basse pression, réduisant l’énergie disponible dans le circuit primaire en cas d’événement affectant son intégrité ;
• une quantité importante d’eau disponible autour du cœur, offrant une forte inertie thermique et une contribution au refroidissement du combustible ;
• une conception privilégiant l’arrêt automatique du réacteur, l’état sûr de l’installation consistant à interrompre la réaction nucléaire dès qu’une anomalie est détectée ;
• un combustible déjà utilisé dans le parc nucléaire français, connu de l’ASNR.
L’un des objectifs du projet est égale - ment d

The examples show us a text that can now be processed using embeddings 

We can now add this text to the others 

In [162]:
df_pages = pd.concat([df_pages, df_consultation], ignore_index = True )
df_pages = df_pages.drop_duplicates(subset=["origin", "page"]).reset_index(drop=True) # if we use several times the concat
print(df_pages.groupby("origin").size())

origin
avis_ASNR_2026.pdf                         4
avis_expertise_ASNR_2026.pdf               6
dossier_concertation_calogena_2026.pdf    39
dtype: int64


We can now apply our function, which will create a document for each source.

In [163]:
documents = pdf_reconstruction(df_pages)

for name, (text, starts) in documents.items():
    print(name, len(text), "characters", len(starts), "pages")

avis_ASNR_2026.pdf 10181 characters 4 pages
avis_expertise_ASNR_2026.pdf 20448 characters 6 pages
dossier_concertation_calogena_2026.pdf 156449 characters 39 pages


The document reconstruction went well

In [164]:
processed_path = Path("data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

with open(processed_path / "documents_01.json", "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

Pandas stored the page numbers as np.int64 rather than int. So we're adding “int” to the pdf_reconstruction function : `starting_page.append((len(text), int(df_documents.loc[i, "page"])))`

It worked, our document containing the three PDFs was successfully saved.

The data engineering portion of the first three documents is complete. We can now move on to chunking and then embedding. We'll cover that in the second notebook.